In [0]:
%pip install  confluent-kafka

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import get_kafka_conf, TOPIC
conf = get_kafka_conf(dbutils)
print("Config set. Topic:", TOPIC)

In [0]:
from confluent_kafka import Consumer

raw_data = spark.read.format('Kafka')\
        .option("kafka.bootstrap.servers", conf["bootstrap.servers"])\
        .option("kafka.security.protocol", "SASL_SSL")\
        .option("kafka.sasl.mechanism",   "PLAIN")\
        .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '   # <-- kafkashaded. prefix
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')\
        .option("subscribe", "post-engagement")\
        .option('startingOffsets', 'earliest')\
            .load()

In [0]:
raw_data.display()

## For replay- First see which timeline needs to pick

In [0]:

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StringType, LongType

schema = (StructType().add("post_id", StringType()).add("type", StringType())
          .add("user_id", StringType()).add("ts", LongType()))

df = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", "post-engagement")
    .option("startingOffsets", "earliest")   # batch: start at beginning
    .option("endingOffsets", "latest")       # read up to current end
    .load())

timeline = (df
    .select("partition", "offset", "timestamp",
            from_json(col("value").cast("string"), schema).alias("e"))
    .select("partition", "offset", "timestamp", "e.post_id", "e.type")
    .orderBy("timestamp"))

display(timeline)

### Here: The same above code we can use to find out the range of offset.. But in real system if we have 500M events, this is not good method. So we use startingTimestamp and endingTimestamp
- lets We already know that this time range is stale
- epoc time 
- start_ms = int(datetime(2026,10,2,19,6,0,tzinfo=timezone.utc).timestamp()*1000)
- end_ms   = int(datetime(2026,10,2,19,8,0,tzinfo=timezone.utc).timestamp()*1000)

- Other method could be- we can pass dict of start offset and end offset

In [0]:
from datetime import datetime, timezone

start_ms = int(datetime(2026,10,2,19,6,0,tzinfo=timezone.utc).timestamp()*1000)
end_ms   = int(datetime(2026,10,2,19,8,0,tzinfo=timezone.utc).timestamp()*1000)

df = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", conf["bootstrap.servers"])
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "PLAIN")
    .option("kafka.sasl.jaas.config",
            f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="{conf["sasl.username"]}" password="{conf["sasl.password"]}";')
    .option("subscribe", "post-engagement")
    .option("startingTimestamp", str(start_ms))
    .option("endingTimestamp",   str(end_ms))
    .load())

from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StringType, LongType
schema = (StructType().add("post_id",StringType()).add("type",StringType())
          .add("user_id",StringType()).add("ts",LongType()))

display(df.select("partition","offset","timestamp",
                  from_json(col("value").cast("string"),schema).alias("e"))
          .select("partition","offset","timestamp","e.post_id","e.type")
          .orderBy("timestamp"))


In [0]:
display(df)